# Pipeline de vendas em tempo quase real — Spark Structured Streaming + Delta Lake

**Arquitetura Medallion:** geração sintética → Bronze (eventos brutos) → Silver (validação, deduplicação e enriquecimento) → Gold (indicadores de negócio).

**Escala:** 1.000.000 vendas, 250.000 clientes, 100.000 produtos. **Microbatches:** a cada 10 segundos. A fonte `rate` emite aproximadamente 1.000 eventos/segundo; o volume total demora cerca de **17 minutos** (mais processamento). O notebook é executável localmente ou em ambiente Spark com Delta Lake. Execute as células em ordem.

> Requisitos: Java 17, Python 3.10+, PySpark e Delta Lake compatíveis. Em produção, substitua a fonte sintética por Kafka/Auto Loader, configure armazenamento distribuído e orquestração de jobs.

## 1. Instalação (execute uma vez, se necessário)

In [ ]:
# Em um ambiente novo, descomente e reinicie o kernel após a instalação.
# %pip install "pyspark==3.5.3" "delta-spark==3.2.1" pandas matplotlib

## 2. Sessão Spark e parâmetros

In [ ]:
import os, time, shutil
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T
from delta import configure_spark_with_delta_pip, DeltaTable

BASE = str(Path.cwd() / "lakehouse_vendas_stream")
TOTAL_VENDAS = 1_000_000
TOTAL_CLIENTES = 250_000
TOTAL_PRODUTOS = 100_000
ROWS_PER_SECOND = 1_000
INTERVALO = "10 seconds"
RESETAR = False  # True APAGA os dados/checkpoints anteriores: apenas demonstração
if RESETAR and Path(BASE).exists(): shutil.rmtree(BASE)

builder = (SparkSession.builder.appName("Vendas-Medallion-Streaming")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "8")
  .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
  .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
  .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

paths = {name: f"{BASE}/{name}" for name in
         ["clientes", "produtos", "bronze", "silver", "gold_produtos", "gold_categorias", "gold_minutos", "checkpoints"]}
print("Spark:", spark.version, "| Diretório:", BASE)

## 3. Dimensões sintéticas (250 mil clientes e 100 mil produtos)
As dimensões são persistidas em Delta e podem ser reutilizadas nas execuções seguintes.

In [ ]:
if not DeltaTable.isDeltaTable(spark, paths["clientes"]):
    clientes = (spark.range(1, TOTAL_CLIENTES + 1)
      .select(F.col("id").cast("long").alias("cliente_id"),
              F.concat(F.lit("Cliente "), F.col("id")).alias("nome_cliente"),
              F.concat(F.lit("cliente"), F.col("id"), F.lit("@exemplo.com")).alias("email"),
              F.element_at(F.array(*[F.lit(x) for x in ["SP", "RJ", "MG", "PR", "RS"]]),
                           (F.col("id") % 5 + 1).cast("int")).alias("uf")))
    clientes.write.format("delta").mode("overwrite").save(paths["clientes"])
if not DeltaTable.isDeltaTable(spark, paths["produtos"]):
    produtos = (spark.range(1, TOTAL_PRODUTOS + 1)
      .select(F.col("id").cast("long").alias("produto_id"),
              F.concat(F.lit("Produto "), F.col("id")).alias("nome_produto"),
              F.element_at(F.array(*[F.lit(x) for x in ["Eletrônicos", "Casa", "Moda", "Esporte", "Livros"]]),
                           (F.col("id") % 5 + 1).cast("int")).alias("categoria"),
              F.round(F.lit(10.0) + (F.col("id") % 990).cast("double"), 2).alias("preco_catalogo")))
    produtos.write.format("delta").mode("overwrite").save(paths["produtos"])
print("Clientes:", spark.read.format("delta").load(paths["clientes"]).count())
print("Produtos:", spark.read.format("delta").load(paths["produtos"]).count())

## 4. Fonte de eventos: geração contínua
`rate` fornece um identificador sequencial (`value`) e timestamp; o filtro limita os IDs a 0…999.999. Valores são determinísticos, facilitando a reexecução e a validação.

In [ ]:
rate = (spark.readStream.format("rate")
    .option("rowsPerSecond", ROWS_PER_SECOND)
    .option("numPartitions", 4).load())
eventos = (rate.filter(F.col("value") < TOTAL_VENDAS)
  .select((F.col("value") + 1).alias("venda_id"),
          F.col("timestamp").alias("data_evento"),
          ((F.col("value") * 37) % TOTAL_CLIENTES + 1).alias("cliente_id"),
          ((F.col("value") * 73) % TOTAL_PRODUTOS + 1).alias("produto_id"),
          ((F.col("value") % 5) + 1).cast("int").alias("quantidade"),
          F.round(F.lit(10.0) + (F.col("value") % 990).cast("double"), 2).alias("preco_unitario"),
          F.lit("simulador_rate").alias("origem"),
          F.current_timestamp().alias("data_ingestao")))
print("Streaming:", eventos.isStreaming)
eventos.printSchema()

## 5. Bronze — ingestão append-only em Delta
O checkpoint guarda offsets e progresso. A execução abaixo é **bloqueante** até gravar os 1 milhão de IDs. A cada 10 s ocorre uma tentativa de microbatch; o tempo real depende do hardware. Se já houver 1 milhão, a etapa é ignorada. Não execute duas instâncias do mesmo pipeline simultaneamente.

In [ ]:
bronze_cp = paths["checkpoints"] + "/bronze"
bronze_atual = (spark.read.format("delta").load(paths["bronze"]).count()
                if DeltaTable.isDeltaTable(spark, paths["bronze"]) else 0)
print("Bronze antes:", bronze_atual)
if bronze_atual < TOTAL_VENDAS:
    q_bronze = (eventos.writeStream.format("delta")
      .outputMode("append").option("checkpointLocation", bronze_cp)
      .trigger(processingTime=INTERVALO).start(paths["bronze"]))
    try:
        while q_bronze.isActive:
            time.sleep(10)
            progresso = q_bronze.lastProgress
            if progresso:
                print("Microbatch:", progresso.get("batchId"),
                      "linhas:", progresso.get("numInputRows"))
            # A contagem é didática; para produção, monitore métricas de progresso.
            if DeltaTable.isDeltaTable(spark, paths["bronze"]):
                n = spark.read.format("delta").load(paths["bronze"]).count()
                print("Bronze acumulada:", n)
                if n >= TOTAL_VENDAS: break
    finally:
        q_bronze.stop()
print("Bronze final:", spark.read.format("delta").load(paths["bronze"]).count())

### Nota de retomada
A fonte `rate` é adequada para demonstrações, **não** para garantia de reprodução de eventos após recriação de checkpoints. Mantenha os checkpoints e os dados juntos. Se desejar recomeçar do zero, ajuste `RESETAR=True` na configuração e execute tudo novamente. Para ingestão real com recuperação robusta, utilize Kafka com retenção adequada.

## 6. Silver — streaming incremental com limpeza e enriquecimento
Leitura streaming da Bronze; `foreachBatch` aplica deduplicação por `venda_id` e `MERGE` idempotente. O enriquecimento usa dimensões Delta estáticas. `availableNow=True` processa tudo que estiver disponível e termina; após novos eventos, execute novamente.

In [ ]:
clientes_dim = spark.read.format("delta").load(paths["clientes"])
produtos_dim = spark.read.format("delta").load(paths["produtos"])

def processar_silver(microbatch, batch_id):
    if microbatch.isEmpty(): return
    limpo = (microbatch
      .filter((F.col("venda_id") > 0) & (F.col("quantidade") > 0) & (F.col("preco_unitario") >= 0))
      .dropDuplicates(["venda_id"]))
    enriquecido = (limpo.join(clientes_dim, "cliente_id", "inner")
      .join(produtos_dim, "produto_id", "inner")
      .withColumn("valor_total", F.round(F.col("quantidade") * F.col("preco_unitario"), 2))
      .withColumn("data_venda", F.to_date("data_evento")))
    if not DeltaTable.isDeltaTable(spark, paths["silver"]):
        enriquecido.limit(0).write.format("delta").mode("overwrite").save(paths["silver"])
    (DeltaTable.forPath(spark, paths["silver"]).alias("t")
       .merge(enriquecido.alias("s"), "t.venda_id = s.venda_id")
       .whenNotMatchedInsertAll().execute())

q_silver = (spark.readStream.format("delta").load(paths["bronze"])
  .writeStream.foreachBatch(processar_silver)
  .option("checkpointLocation", paths["checkpoints"] + "/silver")
  .trigger(availableNow=True).start())
q_silver.awaitTermination()
print("Silver:", spark.read.format("delta").load(paths["silver"]).count())

## 7. Gold — agregações incrementais de negócio
Para manter a execução reprocessável, o `foreachBatch` agrega somente os IDs inéditos em uma tabela de fatos processados. Os indicadores são então **recalculados a partir da Silver**, evitando somas duplicadas em caso de repetição de microbatch. Para 1 milhão de linhas, esse padrão é didático; em escala maior, adote agregações incrementais com estado e/ou Change Data Feed.

In [ ]:
def processar_gold(microbatch, batch_id):
    if microbatch.isEmpty(): return
    # O checkpoint garante que novos lotes disparem a atualização; a fonte de verdade é a Silver.
    base = spark.read.format("delta").load(paths["silver"])
    agregacoes = {
      "gold_produtos": (base.groupBy("produto_id", "nome_produto", "categoria")
         .agg(F.count("*").alias("total_vendas"), F.sum("quantidade").alias("unidades"),
              F.round(F.sum("valor_total"), 2).alias("receita"))),
      "gold_categorias": (base.groupBy("categoria")
         .agg(F.count("*").alias("total_vendas"), F.sum("quantidade").alias("unidades"),
              F.round(F.sum("valor_total"), 2).alias("receita"))),
      "gold_minutos": (base.groupBy(F.date_trunc("minute", "data_evento").alias("minuto"))
         .agg(F.count("*").alias("total_vendas"),
              F.round(F.sum("valor_total"), 2).alias("receita")))
    }
    for nome, df in agregacoes.items():
        # Overwrite com schema preservado: agregados consistentes com o snapshot Silver.
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths[nome])

q_gold = (spark.readStream.format("delta").load(paths["silver"])
  .writeStream.foreachBatch(processar_gold)
  .option("checkpointLocation", paths["checkpoints"] + "/gold")
  .trigger(availableNow=True).start())
q_gold.awaitTermination()
print("Gold atualizada")

## 8. Verificações de qualidade e reconciliação

In [ ]:
bronze = spark.read.format("delta").load(paths["bronze"])
silver = spark.read.format("delta").load(paths["silver"])
assert bronze.count() == TOTAL_VENDAS, "Volume Bronze divergente"
assert silver.count() == TOTAL_VENDAS, "Volume Silver divergente"
assert bronze.select("venda_id").distinct().count() == TOTAL_VENDAS
assert silver.select("venda_id").distinct().count() == TOTAL_VENDAS
assert silver.filter(F.col("valor_total").isNull()).count() == 0
receita_silver = silver.agg(F.sum("valor_total").alias("receita")).first()[0]
receita_gold = (spark.read.format("delta").load(paths["gold_categorias"])
                .agg(F.sum("receita").alias("receita")).first()[0])
assert abs(float(receita_silver) - float(receita_gold)) < 0.01
print("OK: 1 milhão de vendas, IDs únicos, valores válidos e receita reconciliada")

## 9. Análise dos resultados

In [ ]:
print("Top 10 produtos por receita")
spark.read.format("delta").load(paths["gold_produtos"]).orderBy(F.desc("receita")).show(10, truncate=False)
print("Receita por categoria")
spark.read.format("delta").load(paths["gold_categorias"]).orderBy(F.desc("receita")).show(truncate=False)
print("Vendas por minuto")
spark.read.format("delta").load(paths["gold_minutos"]).orderBy("minuto").show(20, truncate=False)

## 10. Monitoramento, exploração e próximos passos

In [ ]:
for camada in ["bronze", "silver", "gold_produtos", "gold_categorias", "gold_minutos"]:
    tabela = DeltaTable.forPath(spark, paths[camada])
    print(camada, "versão mais recente:")
    tabela.history(1).select("version", "timestamp", "operation").show(truncate=False)
# Para conferir checkpoints e arquivos:
print("Diretórios:", sorted(os.listdir(BASE)))

### Considerações de arquitetura

- **Bronze:** mantém os eventos de origem, timestamps e trilha de ingestão, com append e checkpoint.
- **Silver:** aplica regras de qualidade, deduplicação por chave, junções com clientes e produtos, cálculo de valor e MERGE idempotente.
- **Gold:** materializa métricas por produto, categoria e minuto.
- **Latência:** `processingTime="10 seconds"` configura a cadência pretendida dos microbatches, não garante finalização em 10 segundos; os estágios Silver e Gold aqui são executados em sequência após a ingestão para facilitar a aula.
- **Produção:** executar Bronze/Silver/Gold em jobs contínuos independentes; Kafka como fonte durável; monitorar atraso, throughput, falhas e qualidade; usar catálogos, ACLs, otimização de arquivos e políticas de retenção.
- **Observação:** a Gold deste laboratório recalcula o snapshot Silver em cada microbatch e não é eficiente para grandes volumes; a estratégia facilita a correção e evita contagem dupla.

**Exercícios:** introduzir eventos inválidos; comparar tempos de microbatch; adicionar desconto e imposto; gerar dashboard; testar recuperação após reiniciar os streams.